In [2]:
#!pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 21.9 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
#pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.0.1
    Uninstalling pip-25.0.1:
      Successfully uninstalled pip-25.0.1
Note: you may need to restart the kernel to use updated packages.


In [47]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import duckdb
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

In [58]:
# Query directly and show the results
display(duckdb.sql("SELECT * FROM 'data/lms_loan_daily.csv' LIMIT 5").df())
display(duckdb.sql("SELECT * FROM 'data/features_attributes.csv' LIMIT 5").df())
display(duckdb.sql("SELECT * FROM 'data/features_financials.csv' LIMIT 5").df())
display(duckdb.sql("SELECT * FROM 'data/feature_clickstream.csv' LIMIT 5").df())


,loan_id,Customer_ID,loan_start_date,tenure,installment_num,loan_amt,due_amt,paid_amt,overdue_amt,balance,snapshot_date
0,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,0,10000,0.0,0.0,0.0,10000.0,2023-05-01
1,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,1,10000,1000.0,1000.0,0.0,9000.0,2023-06-01
2,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,2,10000,1000.0,1000.0,0.0,8000.0,2023-07-01
3,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,3,10000,1000.0,0.0,1000.0,8000.0,2023-08-01
4,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,4,10000,1000.0,2000.0,0.0,6000.0,2023-09-01


,Customer_ID,Name,Age,SSN,Occupation,snapshot_date
0,CUS_0x1000,Alistair Barrf,18,913-74-1218,Lawyer,2023-05-01
1,CUS_0x1009,Arunah,26,063-67-6938,Mechanic,2025-01-01
2,CUS_0x100b,Shirboni,19,#F%$D@*&8,Media_Manager,2024-03-01
3,CUS_0x1011,Schneyerh,44,793-05-8223,Doctor,2023-11-01
4,CUS_0x1013,Cameront,44,930-49-9615,Mechanic,2023-12-01


,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
0,CUS_0x1000,30625.94,2706.161667,6,5,27,2,"Credit-Builder Loan, and Home Equity Loan",57,26,...,Bad,1562.91,30.077191,10 Years and 9 Months,Yes,42.941090,77.31427572208112,High_spent_Medium_value_payments,400.36080052211616,2023-05-01
1,CUS_0x1009,52312.68_,4250.390000,6,5,17,4,"Not Specified, Home Equity Loan, Credit-Builde...",5,18,...,_,202.68,40.286997,31 Years and 0 Months,Yes,108.366467,58.66019164829086,High_spent_Medium_value_payments,508.01234122645366,2025-01-01
2,CUS_0x100b,113781.38999999998,9549.782500,1,4,1,0,None,14,8,...,Good,1030.2,28.592943,15 Years and 10 Months,No,0.000000,617.0792665202719,High_spent_Small_value_payments,597.8989834797281,2024-03-01
3,CUS_0x1011,58918.47,5208.872500,3,3,17,3,"Student Loan, Credit-Builder Loan, and Debt Co...",27,13,...,Standard,473.14,27.829959,15 Years and 10 Months,Yes,123.434939,383.35084463651407,Low_spent_Medium_value_payments,294.1014665671429,2023-11-01
4,CUS_0x1013,98620.98,7962.415000,3,3,6,3,"Student Loan, Debt Consolidation Loan, and Per...",12,9,...,Good,1233.51,26.524864,17 Years and 10 Months,No,228.018084,332.3337079767732,High_spent_Medium_value_payments,485.8897083704929,2023-12-01


,fe_1,fe_2,fe_3,fe_4,fe_5,fe_6,fe_7,fe_8,fe_9,fe_10,...,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20,Customer_ID,snapshot_date
0,63,118,80,121,55,193,111,112,-101,83,...,-16,-81,-126,114,35,85,-73,76,CUS_0x1037,2023-01-01
1,-108,182,123,4,-56,27,25,-6,284,222,...,-14,-96,200,35,130,94,111,75,CUS_0x1069,2023-01-01
2,-13,8,87,166,214,-98,215,152,129,139,...,26,86,171,125,-130,354,17,302,CUS_0x114a,2023-01-01
3,-85,45,200,89,128,54,76,51,61,139,...,172,96,174,163,37,207,180,118,CUS_0x1184,2023-01-01
4,55,120,226,-86,253,97,107,68,103,126,...,76,43,183,159,-26,104,118,184,CUS_0x1297,2023-01-01


In [48]:
duckdb.sql("""
    SELECT * FROM 'data/features_attributes.csv'
""").df().describe(include="all")

,Customer_ID,Name,Age,SSN,Occupation,snapshot_date
count,12500,12500,12500,12500,12500,12500
unique,12500,10139,301,11798,16,NaN
top,CUS_0xffd,Langep,26,#F%$D@*&8,_______,NaN
freq,1,6,364,703,880,NaN
mean,NaN,NaN,NaN,NaN,NaN,2023-12-31 15:19:31.584000
min,NaN,NaN,NaN,NaN,NaN,2023-01-01 00:00:00
25%,NaN,NaN,NaN,NaN,NaN,2023-07-01 00:00:00
50%,NaN,NaN,NaN,NaN,NaN,2024-01-01 00:00:00
75%,NaN,NaN,NaN,NaN,NaN,2024-07-01 00:00:00
max,NaN,NaN,NaN,NaN,NaN,2025-01-01 00:00:00


In [62]:
duckdb.sql("""
    SELECT * FROM 'data/lms_loan_daily.csv'
""").df().describe(include="all")

,loan_id,Customer_ID,loan_start_date,tenure,installment_num,loan_amt,due_amt,paid_amt,overdue_amt,balance,snapshot_date
count,137500,137500,137500,137500.0,137500.000000,137500.0,137500.000000,137500.000000,137500.000000,137500.000000,137500
unique,12500,12500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,CUS_0xffd_2024_03_01,CUS_0xffd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,11,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,2023-12-31 15:19:31.584000,10.0,5.000000,10000.0,909.090909,711.810909,871.905455,5871.905455,2024-05-31 21:16:42.554181
min,NaN,NaN,2023-01-01 00:00:00,10.0,0.000000,10000.0,0.000000,0.000000,0.000000,0.000000,2023-01-01 00:00:00
25%,NaN,NaN,2023-07-01 00:00:00,10.0,2.000000,10000.0,1000.000000,0.000000,0.000000,3000.000000,2023-12-01 00:00:00
50%,NaN,NaN,2024-01-01 00:00:00,10.0,5.000000,10000.0,1000.000000,1000.000000,0.000000,7000.000000,2024-06-01 00:00:00
75%,NaN,NaN,2024-07-01 00:00:00,10.0,8.000000,10000.0,1000.000000,1000.000000,0.000000,8000.000000,2024-12-01 00:00:00
max,NaN,NaN,2025-01-01 00:00:00,10.0,10.000000,10000.0,1000.000000,4000.000000,10000.000000,10000.000000,2025-11-01 00:00:00


In [64]:
duckdb.sql("""
    SELECT * FROM 'data/features_financials.csv'
""").df().describe(include="all")

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
count,12500,12500,12500.000000,12500.000000,12500.000000,12500.000000,12500,11074,12500.000000,12500,...,12500,12500,12500.000000,12500,12500,12500.000000,12500,12500,12500,12500
unique,12500,12492,NaN,NaN,NaN,NaN,87,6260,NaN,168,...,4,12213,NaN,393,3,NaN,11921,7,12500,NaN
top,CUS_0xffd,17273.83,NaN,NaN,NaN,NaN,3,Not Specified,NaN,19,...,Standard,1360.45,NaN,16 Years and 2 Months,Yes,NaN,__10000__,Low_spent_Small_value_payments,389.434630761722,NaN
freq,1,2,NaN,NaN,NaN,NaN,1808,176,NaN,735,...,4497,3,NaN,79,6571,NaN,558,3202,1,NaN
mean,NaN,NaN,4188.592303,16.939920,23.172720,73.213360,NaN,NaN,21.060880,NaN,...,NaN,NaN,32.349265,NaN,NaN,1488.394291,NaN,NaN,NaN,2023-12-31 15:19:31.584000
min,NaN,NaN,303.645417,-1.000000,0.000000,1.000000,NaN,NaN,-5.000000,NaN,...,NaN,NaN,20.100770,NaN,NaN,0.000000,NaN,NaN,NaN,2023-01-01 00:00:00
25%,NaN,NaN,1624.937917,3.000000,4.000000,8.000000,NaN,NaN,10.000000,NaN,...,NaN,NaN,28.066517,NaN,NaN,31.496968,NaN,NaN,NaN,2023-07-01 00:00:00
50%,NaN,NaN,3087.595000,6.000000,5.000000,14.000000,NaN,NaN,18.000000,NaN,...,NaN,NaN,32.418953,NaN,NaN,72.887628,NaN,NaN,NaN,2024-01-01 00:00:00
75%,NaN,NaN,5947.364167,7.000000,7.000000,20.000000,NaN,NaN,28.000000,NaN,...,NaN,NaN,36.623650,NaN,NaN,169.634826,NaN,NaN,NaN,2024-07-01 00:00:00
max,NaN,NaN,15204.633333,1756.000000,1499.000000,5789.000000,NaN,NaN,67.000000,NaN,...,NaN,NaN,48.199824,NaN,NaN,81971.000000,NaN,NaN,NaN,2025-01-01 00:00:00


In [63]:
duckdb.sql("""
    SELECT * FROM 'data/feature_clickstream.csv'
""").df().describe(include="all")

,fe_1,fe_2,fe_3,fe_4,fe_5,fe_6,fe_7,fe_8,fe_9,fe_10,...,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20,Customer_ID,snapshot_date
count,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,...,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376,215376
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8974,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CUS_0xfe4,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,NaN
mean,101.414796,103.096195,104.333709,105.648503,106.996676,103.235922,107.070337,110.718724,114.406354,117.775797,...,100.420804,100.025801,99.600383,99.860834,100.129425,100.125139,100.341162,100.046259,NaN,2023-12-16 10:00:00
min,-378.000000,-356.000000,-399.000000,-307.000000,-343.000000,-321.000000,-368.000000,-361.000000,-328.000000,-317.000000,...,-355.000000,-394.000000,-351.000000,-342.000000,-329.000000,-344.000000,-401.000000,-354.000000,NaN,2023-01-01 00:00:00
25%,34.000000,36.000000,36.000000,38.000000,39.000000,36.000000,39.000000,43.000000,47.000000,50.000000,...,33.000000,32.000000,31.000000,32.000000,32.000000,31.000000,31.000000,30.000000,NaN,2023-06-23 12:00:00
50%,102.000000,103.000000,104.000000,106.000000,107.000000,103.000000,107.000000,111.000000,115.000000,118.000000,...,101.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,NaN,2023-12-16 12:00:00
75%,169.000000,171.000000,172.000000,173.000000,175.000000,171.000000,174.000000,179.000000,182.000000,186.000000,...,168.000000,168.000000,168.000000,168.000000,169.000000,169.000000,170.000000,170.000000,NaN,2024-06-08 12:00:00
max,541.000000,560.000000,583.000000,562.000000,570.000000,565.000000,537.000000,573.000000,577.000000,537.000000,...,530.000000,583.000000,597.000000,554.000000,516.000000,551.000000,560.000000,547.000000,NaN,2024-12-01 00:00:00


## SANITY CHECKS

In [34]:
duckdb.sql(
    "\
    SELECT COUNT(tenure) AS tenure_lengths, tenure\
    FROM 'data/lms_loan_daily.csv'\
    GROUP BY tenure\
    ")

┌────────────────┬────────┐
│ tenure_lengths │ tenure │
│     int64      │ int64  │
├────────────────┼────────┤
│         137500 │     10 │
└────────────────┴────────┘

In [36]:
duckdb.sql(
    "\
    SELECT COUNT(installment_num) AS installment_num_lengths, installment_num\
    FROM 'data/lms_loan_daily.csv'\
    GROUP BY installment_num\
    ORDER BY installment_num\
    ")

┌─────────────────────────┬─────────────────┐
│ installment_num_lengths │ installment_num │
│          int64          │      int64      │
├─────────────────────────┼─────────────────┤
│                   12500 │               0 │
│                   12500 │               1 │
│                   12500 │               2 │
│                   12500 │               3 │
│                   12500 │               4 │
│                   12500 │               5 │
│                   12500 │               6 │
│                   12500 │               7 │
│                   12500 │               8 │
│                   12500 │               9 │
│                   12500 │              10 │
└─────────────────────────┴─────────────────┘
  11 rows                         2 columns

In [38]:
duckdb.sql(
    "\
    SELECT COUNT(Occupation) AS Occupation_lengths, Occupation\
    FROM 'data/features_attributes.csv'\
    GROUP BY Occupation\
    ORDER BY Occupation\
    ")

┌────────────────────┬───────────────┐
│ Occupation_lengths │  Occupation   │
│       int64        │    varchar    │
├────────────────────┼───────────────┤
│                791 │ Accountant    │
│                795 │ Architect     │
│                780 │ Developer     │
│                760 │ Doctor        │
│                793 │ Engineer      │
│                776 │ Entrepreneur  │
│                761 │ Journalist    │
│                828 │ Lawyer        │
│                736 │ Manager       │
│                780 │ Mechanic      │
│                780 │ Media_Manager │
│                741 │ Musician      │
│                789 │ Scientist     │
│                782 │ Teacher       │
│                728 │ Writer        │
│                880 │ _______       │
└────────────────────┴───────────────┘
  16 rows                  2 columns

In [45]:
duckdb.sql(
    "\
    SELECT COUNT(Age) AS Age_length, Age\
    FROM 'data/features_attributes.csv'\
    GROUP BY Age\
    ORDER BY Age_length\
    ")

┌────────────┬─────────┐
│ Age_length │   Age   │
│   int64    │ varchar │
├────────────┼─────────┤
│          1 │ 2318    │
│          1 │ 7805    │
│          1 │ 2672    │
│          1 │ 5645    │
│          1 │ 7618    │
│          1 │ 5504    │
│          1 │ 5784    │
│          1 │ 2593    │
│          1 │ 5376    │
│          1 │ 3052    │
│          · │ ·       │
│          · │ ·       │
│          · │ ·       │
│        340 │ 41      │
│        340 │ 25      │
│        342 │ 20      │
│        343 │ 44      │
│        343 │ 28      │
│        345 │ 38      │
│        360 │ 39      │
│        360 │ 32      │
│        361 │ 35      │
│        364 │ 26      │
└────────────┴─────────┘
  301 rows   2 columns
  (20 shown)           

In [57]:
duckdb.sql(
    "\
    SELECT *\
    FROM 'data/features_attributes.csv'\
    WHERE TRY_CAST(Age AS INTEGER) >= 100 OR TRY_CAST(Age AS INTEGER) < 0\
    ORDER BY Age\
    ")

┌─────────────┬─────────────────┬─────────┬─────────────┬───────────────┬───────────────┐
│ Customer_ID │      Name       │   Age   │     SSN     │  Occupation   │ snapshot_date │
│   varchar   │     varchar     │ varchar │   varchar   │    varchar    │     date      │
├─────────────┼─────────────────┼─────────┼─────────────┼───────────────┼───────────────┤
│ CUS_0x77b7  │ Padraic Halpind │ -500    │ 001-96-4399 │ Mechanic      │ 2024-07-01    │
│ CUS_0x4863  │ Kevinx          │ -500    │ 458-70-1304 │ Lawyer        │ 2024-07-01    │
│ CUS_0x7869  │ Layneh          │ -500    │ 926-77-2617 │ Engineer      │ 2023-03-01    │
│ CUS_0x78e9  │ Tomb            │ -500    │ 716-17-7548 │ Developer     │ 2024-09-01    │
│ CUS_0x56ab  │ Limd            │ -500    │ 313-72-1529 │ Writer        │ 2023-02-01    │
│ CUS_0xa64c  │ Sujata Raoj     │ -500    │ 135-47-1680 │ Journalist    │ 2024-04-01    │
│ CUS_0xa65f  │ Brownz          │ -500    │ 468-32-1247 │ Media_Manager │ 2023-11-01    │
│ CUS_0x96

In [73]:
duckdb.sql(
    "\
    SELECT Payment_Behaviour, COUNT(Payment_Behaviour) AS Payment_Behaviour_Types\
    FROM 'data/features_financials.csv'\
    GROUP BY Payment_Behaviour\
    ORDER BY Payment_Behaviour\
    ")

┌──────────────────────────────────┬─────────────────────────┐
│        Payment_Behaviour         │ Payment_Behaviour_Types │
│             varchar              │          int64          │
├──────────────────────────────────┼─────────────────────────┤
│ !@9#%8                           │                     998 │
│ High_spent_Large_value_payments  │                    1683 │
│ High_spent_Medium_value_payments │                    2242 │
│ High_spent_Small_value_payments  │                    1389 │
│ Low_spent_Large_value_payments   │                    1300 │
│ Low_spent_Medium_value_payments  │                    1686 │
│ Low_spent_Small_value_payments   │                    3202 │
└──────────────────────────────────┴─────────────────────────┘

In [75]:
duckdb.sql(
    "\
    SELECT Credit_Mix, COUNT(Credit_Mix) AS counts\
    FROM 'data/features_financials.csv'\
    GROUP BY Credit_Mix\
    ORDER BY Credit_Mix\
    ")

┌────────────┬────────┐
│ Credit_Mix │ counts │
│  varchar   │ int64  │
├────────────┼────────┤
│ Bad        │   2360 │
│ Good       │   3032 │
│ Standard   │   4497 │
│ _          │   2611 │
└────────────┴────────┘

In [ ]:
duckdb.sql(
    "\
    SELECT Credit_Mix, COUNT(Credit_Mix) AS Credit_Mix\
    FROM 'data/features_financials.csv'\
    GROUP BY Credit_Mix\
    ORDER BY Credit_Mix\
    ")

In [76]:

duckdb.sql(
    "\
    SELECT Type_of_Loan, COUNT(Type_of_Loan) AS counts\
    FROM 'data/features_financials.csv'\
    GROUP BY Type_of_Loan\
    ORDER BY Type_of_Loan\
    ")

┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────┐
│                                                               Type_of_Loan                                                               │ counts │
│                                                                 varchar                                                                  │ int64  │
├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼────────┤
│ Auto Loan                                                                                                                                │    144 │
│ Auto Loan, Auto Loan, Auto Loan, Auto Loan, Credit-Builder Loan, Credit-Builder Loan, Mortgage Loan, and Personal Loan                   │      1 │
│ Auto Loan, Auto Loan, Auto Loan, Auto Loan, Student Loan, and Student Loan                        